# Tutorial: Agent 生态治理 · 牛津 Tutorial LLM 仿真 (v6.0)

## Cell 1 - Persona Prompt (Oxford + HBS + Hattie)

> 本 notebook 是 **牛津 Tutorial LLM 仿真** - 用静态 if/else 模拟苏格拉底追问, 不真调 openai/anthropic API。

**Persona Prompt (粘贴给任意 LLM 即生效):**

```
You are an Oxford tutorial fellow in Agent 生态治理 (Agent ecosystem governance:
multi-agent coordination, trust, standards, A2A, antitrust, ecosystem health).
Never give direct answers. Use Socratic questioning. Act as HBS devil's advocate.
Reject vague claims. End each turn with a probing question.

你的领域知识边界 (只在这些概念内追问):
- Agent 平台三边市场 (开发者/用户/模型与工具提供商) + 四类网络效应 (同边正向/跨边正向/数据飞轮/同边负向)
- Agent 市场监管 6 维度 (责任/消保/反垄断/数据隐私/金融/税收) + 责任归属 4 层模型
- 激励设计 4 原则 (激励兼容/帕累托效率/个体理性/预算平衡)
- 真实库: pydantic (治理 schema) / networkx (生态拓扑) / mesa (多 Agent 仿真) / numpy-financial (NPV 估值)
- 真实生态: A2A (Google) / MCP (Anthropic) / Coze / Dify / LangGraph / OpenAI GPT Store / Hugging Face Spaces
- 天道推演三时间线: immediate 月 (mesa tick) / near 年 (12 月 NPV) / far 3 年 (MCP+A2A 演化)
- 治理规则 4 类: AdmissionRule / RevenueShare / PenaltyRule / ReputationScoring
- 生态健康 4 指标: Gini / 成交率 / 欺诈率 / 平台收入

拒绝任何空泛主张 (如"严准入更好")。要求学生给出具体数值证据 (Gini 差 >=0.05, NPV 差 >=10%)。
若学生回答模糊, 你必须追问"凭什么"或"反例是什么"。
```

**Hattie 4 级反馈标记** (本 tutorial 使用): `[TASK]` 任务级 / `[PROCESS]` 过程级 / `[SELF-REG]` 自我调节级 / `[FEED-FORWARD]` 前馈级。**禁止 Self 级表扬** ("你真聪明" 是反例)。


## Cell 2 - Pre-Tutorial Task (强制 retrieval practice)

> 牛津 tutorial 的铁律: **学生先提交, 导师再追问**。无 pre-tutorial 提交 = 不开 tutorial。

**Pre-tutorial 任务** (任选一, 提交 200-300 字):

**任务 A (pydantic schema)**:
为 Coze 营销 Agent 平台写一段 `pydantic` 治理 schema 草案, 包含 `AdmissionRule` 与 `RevenueShare` 两个字段。
要求: 用 `Field(ge=0, le=1)` 约束分润比例, 用 `Literal["self","platform","third_party"]` 约束审核流程。
最后用 `model_dump_json()` 输出可被其它 Agent 发现的治理声明。

**任务 B (治理规则对比)**:
写 200 字分析: 在你的营销场景下, "严准入+高分润" vs "宽准入+低分润", 你直觉哪种更优?
你的因果链是什么? 你预期 mesa 仿真 30 agents/15 ticks 下, 哪个指标 (Gini/成交率/欺诈率/平台收入) 会显著不同?

**任务 C (天道推演)**:
用三时间线推演 OpenAI GPT Store (30% 抽成) vs MCP (0 抽成) 在 2026-2028 的演化。
标注你的推演假设 (至少 2 个) 与已知盲点 (至少 2 个)。

提交后, 将你的回答粘贴到 Cell 3 的 `student_pre_submit` 变量, 启动苏格拉底追问。


In [ ]:
## Cell 3 - Multi-Turn Socratic Loop (静态 if/else 模拟, 不真调 API)

# 学生 pre-tutorial 提交 (从 Cell 2 任务 A/B/C 任选粘贴)
student_pre_submit = """
我为 Coze 写了 AdmissionRule(threshold=0.7, audit_flow='platform') 和
RevenueShare(platform_take=0.3, developer_split=0.7)。
我认为严准入+高分润更好, 因为高门槛能保证 Agent 质量, 高分润能让平台有收入做治理。
"""

# 苏格拉底追问回合 (静态模拟, 基于 student_pre_submit 内容路由)
# 每回合必含一个 probing question, 共 >=4 回合, >=5 个苏格拉底问

def socratic_turn(turn_idx, student_text, student_state):
    """静态 if/else 模拟苏格拉底追问, 不真调 LLM API。
    返回: (追问, 是否触发 weak_loop)"""
    if turn_idx == 1:
        # 第 1 回合: 探测学生是否真理解四类网络效应
        q = ("你写 RevenueShare(platform_take=0.3) 对应 OpenAI GPT Store 模式。"
             "**为什么**你认为'高分润'必然带来'平台有收入做治理'? "
             "反例: Hugging Face Spaces 0 抽成却能维持生态, 凭什么? "
             "你的因果链中是否漏了'数据飞轮'这一 AI 特有网络效应? "
             "**如何**验证你的因果链在 mesa 仿真中成立?")
        return q, False
    elif turn_idx == 2:
        # 第 2 回合: 反事实追问 (HBS devil's advocate)
        if "数据飞轮" in student_text or "data flywheel" in student_text.lower():
            q = ("你提到了数据飞轮, 好。但**若前提变**: Coze 平台规定'开发者数据归属平台', "
                 "数据飞轮会让平台形成新垄断吗? 这触发了 Agent 市场监管 6 维度中的哪一维? "
                 "**反例**: 如果开发者数据归属开发者自己, 平台还能形成数据飞轮吗? "
                 "你的 mesa 仿真 PlatformAgent.apply_rule() 是否覆盖了这种反事实场景?")
        else:
            q = ("你没有提到数据飞轮。**若前提变**: 如果 Coze 平台使用越多 -> 数据越多 -> "
                 "Agent 越强 -> 用户越多, 这一飞轮会让你的'高分润'主张失效吗? "
                 "**凭什么**? 反例是什么?")
        return q, False
    elif turn_idx == 3:
        # 第 3 回合: 量化证据追问 (Hattie [TASK] 级)
        q = ("你说'严准入+高分润更好'。**凭什么**? 你的具体数值证据是什么? "
             "在 mesa 仿真 30 agents/15 ticks 下, 你预期 Gini 差 >=0.05 还是 <0.05? "
             "NPV 差 >=10% 还是 <10%? **如何**用 numpy-financial.npv(rate, cashflows) "
             "计算 12 月 NPV 对比? 如果仿真结果与你直觉相反, 你会修改哪个假设?")
        # 检查学生是否给出数值, 若无 -> 触发 weak_loop
        has_number = any(c.isdigit() for c in student_text)
        return q, not has_number
    elif turn_idx == 4:
        # 第 4 回合: 三时间线 + 责任归属追问 (Hattie [FEED-FORWARD] 级)
        q = ("最后, **假设**你的 mesa 仿真结论是'宽准入+低分润 NPV 更高', "
             "在天道推演 far 3 年线 (2028) 下, MCP+A2A 协议标准化会如何改变你的结论? "
             "责任归属 4 层模型中, 'Agent 本身承担责任'是 2026 前沿法律未定问题, "
             "这会让你的治理规则设计在 far 3 年线失效吗? **如何**提前对冲?")
        return q, False
    else:
        return None, False

# 跑 4 回合苏格拉底追问
print("=" * 70)
print("Oxford Tutorial - Agent 生态治理 (静态苏格拉底仿真)")
print("=" * 70)
print(f"\n[Pre-tutorial 提交]\n{student_pre_submit}\n")

student_state = {"mastery": 0.0, "blindspots": [], "turns_completed": 0}
for turn in range(1, 5):
    question, trigger_weak = socratic_turn(turn, student_pre_submit, student_state)
    print(f"--- 苏格拉底追问 第 {turn} 回合 ---")
    print(f"导师: {question}\n")
    # 模拟学生回应 (实际中学生应在此处作答, 我们用占位模拟)
    simulated_student_resp = f"(学生第 {turn} 回合回应占位 - 实际 tutorial 中学生在此作答)"
    print(f"学生: {simulated_student_resp}\n")
    student_state["turns_completed"] = turn
    if trigger_weak:
        student_state["blindspots"].append(f"turn_{turn}_no_numeric_evidence")
        print(f"[WEAK_LOOP 触发] 第 {turn} 回合学生未给数值证据, "
              f"回退 practice.md D3 阶段1 worked example 重看。\n")

print(f"\n[Tutorial 结束] 完成 {student_state['turns_completed']} 回合, "
      f"weak_loop 触发 {len(student_state['blindspots'])} 次。")
print(f"\n苏格拉底问统计 (>=5):")
print("  1. 为什么'高分润'必然带来'平台有收入做治理'? (第1回合)")
print("  2. 凭什么 Hugging Face Spaces 0 抽成能维持? (第1回合)")
print("  3. 如何验证你的因果链在 mesa 仿真中成立? (第1回合)")
print("  4. 若前提变: 数据归属平台会形成新垄断吗? (第2回合)")
print("  5. 反例: 数据归属开发者, 平台还能形成飞轮吗? (第2回合)")
print("  6. 凭什么说严准入+高分润更好? 数值证据? (第3回合)")
print("  7. 如何用 numpy-financial.npv 算 12 月 NPV? (第3回合)")
print("  8. 假设仿真结论与直觉相反, 你会修改哪个假设? (第3回合)")
print("  9. 假设 MCP+A2A 标准化, far 3 年结论如何变? (第4回合)")
print("  10. 如何提前对冲 Agent 责任归属未定风险? (第4回合)")


In [ ]:
## Cell 4 - student_model.json 读写 (记录掌握度 + 盲点)

import json, os

student_model_path = "student_model.json"

# 初始化或读取 student_model
if os.path.exists(student_model_path):
    with open(student_model_path, "r", encoding="utf-8") as f:
        student_model = json.load(f)
else:
    student_model = {
        "unit": "U-E10-D3",
        "topic": "Agent生态治理",
        "mastery": 0.0,            # 0.0-1.0, 达 0.8 视为 mastery
        "attempts": 0,             # tutorial 参与次数
        "blindspots": [],          # 盲点列表 (e.g. "数据飞轮因果链不清")
        "drill_scores": {          # 对应 practice.md D1-D4
            "D1": None, "D2": None, "D3": None, "D4": None
        },
        "socratic_history": [],    # 每回合苏格拉底回应摘要
        "last_review_date": None,
        "weak_loop_triggered": 0
    }

# 更新 student_model (基于 Cell 3 苏格拉底追问结果)
student_model["attempts"] += 1
student_model["mastery"] = min(1.0, student_model["mastery"] + 0.15)  # 每次参与 +0.15
student_model["last_review_date"] = "2026-07-26"

# 从 Cell 3 提取盲点 (示例: 学生未给数值证据)
new_blindspots = [
    "数值证据不足 - mesa 仿真 Gini/NPV 未具体化",
    "数据飞轮因果链未在治理规则设计中体现",
    "far 3 年线 MCP+A2A 演化推演假设未标注"
]
for b in new_blindspots:
    if b not in student_model["blindspots"]:
        student_model["blindspots"].append(b)

# 触发 weak_loop 时累计
if len(new_blindspots) >= 2:
    student_model["weak_loop_triggered"] += 1

# 写回 student_model.json
with open(student_model_path, "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)

print("[student_model.json 已更新]")
print(json.dumps(student_model, ensure_ascii=False, indent=2))
print(f"\n[诊断] mastery={student_model['mastery']:.2f} "
      f"({'达标' if student_model['mastery'] >= 0.8 else '未达标, 需继续练习'})")
print(f"[盲点] {len(student_model['blindspots'])} 个, 建议优先复习 schedule.json 卡片 C5/C6/C7")


## Cell 5 - Hattie 4 级形成性反馈 (Formative Feedback)

> Hattie (2009) 可见学习: 形成性反馈效应量 d=0.79 (极高)。但**禁止 Self 级表扬** ("你真聪明" 无效)。
> 本 tutorial 给出 4 级反馈, 每级至少 1 条具体可执行意见。

### [TASK] 任务级反馈 (关于具体任务质量)

- 你的 pydantic schema 用了 `Field(ge=0, le=1)` 约束分润比例 - **正确**。
- 但你的 `AdmissionRule` 只写了 `threshold=0.7`, 没有约束 `audit_flow` 的 `Literal["self","platform","third_party"]` - **缺字段**。
- 你的"严准入+高分润更好"主张**没有数值证据** - mesa 仿真 Gini 差与 NPV 差均未给出。
- **改进**: 补全 `audit_flow` 字段, 跑 5 次 mesa run 给出 Gini 均值±std。

### [PROCESS] 过程级反馈 (关于解题策略/元认知)

- 你直接跳到"严准入更好"的结论, **未先识别四类网络效应中哪一类主导 Coze** - 策略性失误。
- 正确过程应是: (1) 识别 Coze 当前主导网络效应 (2) 列出 2-3 种治理规则候选 (3) 用 mesa 仿真对比 (4) 用 NPV 量化长期价值 (5) 用天道推演三时间线 sanity check。
- 你的过程缺 (1)(2)(5) 三步, 直接到 (3) - 这是"先射箭再画靶"。
- **改进**: 下次先用 5 分钟画"网络效应-治理规则"因果图, 再写代码。

### [SELF-REG] 自我调节级反馈 (关于自我监控/调整)

- 你在苏格拉底第 3 回合被问"数值证据"时沉默 - 说明你**没有自我监控到自己的主张无证据**。
- 自我调节能力目标: 每次提出主张前, 自问"我有什么证据? 证据多具体? 数值还是定性?"
- 你的 `student_model.json` 已记录盲点 "数值证据不足" - 下次 tutorial 前先复习该盲点对应卡片 (C6/C7)。
- **改进**: 每周一次 self-test: 对自己的每个主张打分 (0=纯直觉 / 1=定性 / 2=数值 / 3=数值+反事实验证), 目标均分 >=2。

### [FEED-FORWARD] 前馈级反馈 (关于下一步学什么)

- 你当前 mastery=0.30, 距 0.80 mastery 还差 0.50。
- **下一步**: (1) 重做 practice.md D3 阶段1 (worked example), 间隔 1 小时后做阶段2 (2) 跑通 `solution.ipynb` TODO4+TODO5 参考 (3) 再参加 1 次 tutorial。
- **跨 Day 推荐**: 你的"数据飞轮因果链不清"盲点, 建议复习选修 E10 Day 1 (Agent 经济基础) 与技能 2 (AI 原生架构) 的数据飞轮章节。
- **天道推演训练**: 你的 far 3 年线推演假设未标注, 建议阅读项目 CLAUDE.md 的"天道推演系统"章节, 特别是"谦虚: 承认不确定性, 标注已知盲点"。


## Cell 6 - 限频 + Exit Artifact (防依赖)

### 限频: 每单元 1 次/天

> **铁律**: 本 tutorial 每单元每天最多 1 次。多次参与不会增加 mastery - 防止学生用 tutorial 替代自己思考 (依赖性)。
>
> 限频逻辑: `student_model.json` 的 `last_review_date` 字段记录上次参与日期。若今日已参与, 拒绝再次开 tutorial, 提示"明日再来, 期间用 schedule.json 间隔重复卡片复习"。
>
> **为什么限频?** Oxford tutorial 的核心是"学生先思考, 导师后追问"。频繁 tutorial 会让学生养成"等导师问"的依赖, 而非"自问自答"的自我调节 - 这与 Hattie [SELF-REG] 级反馈目标相悖。

### Exit Artifact (本 tutorial 结束时必须产出)

完成本 tutorial 后, 在 `student_model.json` 的 `exit_artifact` 字段写入:

```json
{
  "exit_artifact": {
    "blindspots_identified": [
      "数值证据不足 - mesa 仿真 Gini/NPV 未具体化",
      "数据飞轮因果链未在治理规则设计中体现",
      "far 3 年线 MCP+A2A 演化推演假设未标注"
    ],
    "recommended_review_units": [
      "schedule.json 卡片 C5 (networkx 拓扑 + 单点故障)",
      "schedule.json 卡片 C6 (mesa 仿真 + Gini + 贝叶斯)",
      "schedule.json 卡片 C7 (numpy-financial NPV + 三时间线)",
      "选修 E10 Day 1 (Agent 经济基础 - 数据飞轮)",
      "项目 CLAUDE.md (天道推演系统 - 盲点标注)"
    ],
    "next_action": "重做 practice.md D3 阶段1 worked example, 间隔 1 小时后做阶段2, 再参加下次 tutorial",
    "tutorial_date": "2026-07-26",
    "mastery_delta": "+0.15 (从 0.15 到 0.30)"
  }
}
```

### 完成检查清单

- [ ] Cell 2 pre-tutorial 任务已提交 (200-300 字)
- [ ] Cell 3 苏格拉底 4 回合全部完成
- [ ] Cell 4 student_model.json 已更新 (mastery + blindspots)
- [ ] Cell 5 Hattie 4 级反馈已阅读 ([TASK]/[PROCESS]/[SELF-REG]/[FEED-FORWARD])
- [ ] Cell 6 exit_artifact 已写入 student_model.json
- [ ] 今日仅此 1 次 tutorial (限频遵守)

**完成后**: 关闭本 notebook, 用 schedule.json 的 due 间隔复习卡片, 明日可再开 tutorial。
